# IBM RAG Updated (Clean)

Refactored for readability: section comments moved into Markdown, outputs removed, and execution state reset.

In [ ]:
%pip install -Uq "docling>=2.0.0" 
%pip install -Uq "docling-core>=2.0.0" 
%pip install -Uq "langchain>=0.3.0" 
%pip install -Uq "langchain-core>=0.3.0" 
%pip install -Uq "langchain-community>=0.3.0" 
%pip install -Uq "langchain-ollama>=0.2.0" 
%pip install -Uq "langchain-chroma>=0.1.2" 
%pip install -Uq "langchain-classic" e
%pip install -Uq "chromadb>=0.5.0" 
%pip install -Uq "flashrank>=0.2.9" 
%pip install -Uq "rank-bm25" 
%pip install -Uq "ollama>=0.4.0" 
%pip install -Uq "Pillow>=10.0.0" 
%pip install -Uq "transformers>=4.40.0" 
%pip install -Uq "sentence-transformers>=3.0.0" 
%pip install -Uq 
%pip install -Uq "datasets" 
%pip install -Uq "pandas"
%pip install -Uq "gradio"

## MULTIMODAL RAG PIPELINE — Complete Clean Version

- Stack : Docling · ChromaDB · BM25 · FlashRank · Ollama (llava + llama2)
- New   : Semantic Layer (HyDE + Query Expansion)
- Multi-page Table Stitching


In [1]:
# ── Install (run once) ────────────────────────────────────────────────────────
# pip install -Uq "docling>=2.0.0" "langchain>=0.3.0" "langchain-core>=0.3.0"
#             "langchain-community>=0.3.0" "langchain-ollama>=0.2.0"
#             "langchain-chroma>=0.1.2" "chromadb>=0.5.0" "ollama>=0.4.0"
#             "transformers>=4.40.0" "sentence-transformers>=3.0.0"
#             "pillow>=10.0.0" "rank_bm25" "flashrank>=0.2.9"
#             "langchain-classic" "datasets" "pandas"
#
# ollama pull llava
# ollama pull llama2          # or llama3.2
# ollama pull nomic-embed-text


# =============================================================================
# CELL 1 │ Imports & Configuration
# =============================================================================

import io
import json
import logging
import re
import itertools
from difflib import SequenceMatcher
from typing import Optional

import PIL.Image
import PIL.ImageOps

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# ── Ollama endpoints ──────────────────────────────────────────────────────────
OLLAMA_BASE_URL  = "http://localhost:11434"
VISION_MODEL     = "llava"             # ollama pull llava
CHAT_MODEL       = "llama3.2"            # ollama pull llama2 / llama3.2
EMBEDDING_MODEL  = "nomic-embed-text"  # ollama pull nomic-embed-text

# ── Retrieval tuning ──────────────────────────────────────────────────────────
BM25_TOP_K      = 15
DENSE_TOP_K     = 15
RERANKER_TOP_N  = 5

# ── Chunking (tuned for financial survey PDFs) ────────────────────────────────
PARENT_CHUNK_SIZE    = 1500
PARENT_CHUNK_OVERLAP = 100
CHILD_CHUNK_SIZE     = 350
CHILD_CHUNK_OVERLAP  = 40

# ── Paths ─────────────────────────────────────────────────────────────────────
PDF_SOURCES = [
    r"C:\Users\Rutvik Rathva\Downloads\Intern 3.12\Dataset_removed-pages-deleted.pdf",
]
CHROMA_PERSIST_DIR = "./chroma_rag_final"

DEFAULT_QUERY = (
    "What is the real GDP growth rate for FY26 "
    "according to the First Advance Estimates?"
)

In [ ]:
%pip install -Uq langchain-ollama

## CELL 2 │ Load Ollama Models

In [2]:
from langchain_ollama import OllamaEmbeddings, ChatOllama

def load_models():
    logger.info("Loading Ollama models...")
    embeddings_model = OllamaEmbeddings(
        model=EMBEDDING_MODEL,
        base_url=OLLAMA_BASE_URL,
    )
    logger.info(f"  ✓ Embeddings : {EMBEDDING_MODEL}")

    chat_model = ChatOllama(
        model=CHAT_MODEL,
        base_url=OLLAMA_BASE_URL,
        temperature=0.0,
        num_ctx=8192,
    )
    logger.info(f"  ✓ Chat model : {CHAT_MODEL}")
    return embeddings_model, chat_model

embeddings_model, chat_model = load_models()

c:\Users\Rutvik Rathva\Downloads\Modular Code\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
02:34:00 [INFO] Loading Ollama models...
02:34:01 [INFO]   ✓ Embeddings : nomic-embed-text
02:34:02 [INFO]   ✓ Chat model : llama3.2


## CELL 3 │ Image Classification Helper

In [3]:
import ollama as ollama_client

def classify_image(image: PIL.Image.Image) -> str:
    """
    Classify image type using aspect ratio heuristic + quick llava check.
    Returns one of: dual_chart, colour_table, single_chart, equation, other
    """
    width, height = image.size
    aspect = width / height if height > 0 else 1.0

    if aspect > 1.6:
        image_rgb = image.convert("RGB")
        buf = io.BytesIO()
        image_rgb.save(buf, format="PNG")
        img_bytes = buf.getvalue()
        try:
            resp = ollama_client.chat(
                model=VISION_MODEL,
                messages=[{
                    "role": "user",
                    "content": (
                        "Look at this image carefully. Reply with ONLY one word:\n"
                        "- 'dual' if it shows two charts/graphs side by side\n"
                        "- 'table' if it shows a data table with coloured cells\n"
                        "- 'chart' if it shows a single chart or graph\n"
                        "- 'equation' if it shows mathematical formulas\n"
                        "- 'other' for anything else"
                    ),
                    "images": [img_bytes],
                }],
                options={"num_predict": 5, "temperature": 0},
            )
            answer = resp["message"]["content"].strip().lower()
            if "dual"  in answer: return "dual_chart"
            if "table" in answer: return "colour_table"
            if "chart" in answer: return "single_chart"
            if "equat" in answer: return "equation"
        except Exception:
            pass
        return "dual_chart"

    if 0.7 < aspect < 1.6:
        return "single_chart"

    return "other"

## CELL 4 │ Fix 1 — Colour-Table Extraction

In [4]:
COLOUR_TABLE_PROMPT = """\
This image shows a data table from an Indian economic report.
The table uses COLOUR CODING to show performance relative to historical averages:
  GREEN  or TEAL  cells  = value is ABOVE the historical average (POSITIVE signal)
  RED    or PINK  cells  = value is BELOW the historical average (NEGATIVE signal)
  YELLOW or AMBER cells  = neutral / informational value
  WHITE          cells  = no special significance

Your task — extract ALL data with colour context:

Step 1: State the table title exactly.
Step 2: List all column headers exactly as shown.
Step 3: For EVERY data cell, output:
  <Row label> | <Column header>: <value> [ABOVE AVG / BELOW AVG / NEUTRAL / N/A]

Output format (plain text, no JSON):
TABLE TITLE: <exact title>
HEADERS: <col1> | <col2> | <col3> | ...
DATA:
<row1 label> | <col1>: <val> [ABOVE/BELOW/NEUTRAL] | <col2>: <val> [ABOVE/BELOW/NEUTRAL] | ...
<row2 label> | ...

IMPORTANT:
- Extract EVERY number visible, including negative values.
- Do NOT skip any rows or columns.
- If a cell says 'NA' or is empty, write NA.
- Preserve exact decimal values (e.g. 33.3 not ~33).
"""


def extract_colour_table(image: PIL.Image.Image) -> str:
    image = PIL.ImageOps.exif_transpose(image) or image
    image = image.convert("RGB")
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    img_bytes = buf.getvalue()
    try:
        resp = ollama_client.chat(
            model=VISION_MODEL,
            messages=[{"role": "user", "content": COLOUR_TABLE_PROMPT, "images": [img_bytes]}],
            options={"num_predict": 1500, "temperature": 0},
        )
        return resp["message"]["content"].strip()
    except Exception as e:
        logger.warning(f"  Colour table extraction failed: {e}")
        return "[Colour table extraction unavailable]"

## CELL 5 │ Fix 2 — Dual-Chart Extraction

In [5]:
DUAL_CHART_PROMPT = """\
This image contains EXACTLY TWO charts displayed side by side.

Extract each chart COMPLETELY AND SEPARATELY. Never mix data between them.

LEFT CHART:
  Title: <exact title text from the left chart>
  Type: <bar / line / mixed / scatter>
  X-axis label: <label or time period>
  Y-axis label: <unit e.g. "% YoY growth" or "Policy rate (%)">
  Series:
    <series name 1>: <period/category>=<value>, <period/category>=<value>, ...
    <series name 2>: <period/category>=<value>, ...
  Key values: <most important 2-3 data points with labels>
  Source: <source note if visible>

RIGHT CHART:
  Title: <exact title text from the right chart>
  Type: <bar / line / mixed / scatter>
  X-axis label: <label or time period>
  Y-axis label: <unit>
  Series:
    <series name 1>: <period/category>=<value>, ...
  Key values: <most important 2-3 data points with labels>
  Source: <source note if visible>

RULES:
- Extract EVERY data point and legend label visible in each chart.
- Preserve all numeric values exactly as shown (e.g. 7.4 not "about 7").
- If a data label is visible ON the chart, include it.
- Keep LEFT and RIGHT sections completely separate.
"""


def extract_dual_chart(image: PIL.Image.Image) -> str:
    image = PIL.ImageOps.exif_transpose(image) or image
    image = image.convert("RGB")
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    img_bytes = buf.getvalue()
    try:
        resp = ollama_client.chat(
            model=VISION_MODEL,
            messages=[{"role": "user", "content": DUAL_CHART_PROMPT, "images": [img_bytes]}],
            options={"num_predict": 1500, "temperature": 0},
        )
        return resp["message"]["content"].strip()
    except Exception as e:
        logger.warning(f"  Dual chart extraction failed: {e}")
        return "[Dual chart extraction unavailable]"

## CELL 6 │ Standard Single-Chart & Equation Extraction

In [6]:
SINGLE_CHART_PROMPT = """\
You are a financial data extraction specialist analyzing a document image.

Step 1 — Identify: chart, table, diagram, infographic, equation, or photo.

Step 2 — Extract ALL visible data into this exact JSON:
{
  "type": "<chart|table|diagram|infographic|equation|photo|other>",
  "title": "<exact title or null>",
  "subtitle": "<subtitle/date range or null>",
  "data": {
    "chart_type": "<bar|line|pie|scatter|mixed|null>",
    "x_axis_label": "<label or null>",
    "y_axis_label": "<label or null>",
    "series": [
      {"name": "<series name>", "values": {"<x>": "<y with unit>", ...}}
    ],
    "table_headers": ["<col1>", ...],
    "table_rows": [{"<col1>": "<val>", ...}]
  },
  "key_facts": [
    "<Fact with number and label e.g. GDP FY26E: 7.4%>",
    ...
  ],
  "source_note": "<source text or null>"
}

Rules:
- Extract EVERY number and label.
- Return ONLY valid JSON, no markdown fences.
- If photo/logo with no data: type='photo', key_facts=[].
"""

EQUATION_PROMPT = """\
This image contains mathematical equations or formulas from an economics/statistics paper.

Extract EVERY equation exactly as written:
1. Write each equation on a separate line
2. Use standard notation: subscripts as _x, superscripts as ^x, Greek letters spelled out
3. Include variable definitions if shown
4. Include any surrounding explanatory text

Format:
EQUATION 1: <equation>
EQUATION 2: <equation>
DEFINITIONS:
  <variable>: <definition>
CONTEXT: <brief description of what these equations model>
"""


def extract_single_chart(image: PIL.Image.Image) -> str:
    image = PIL.ImageOps.exif_transpose(image) or image
    image = image.convert("RGB")
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    img_bytes = buf.getvalue()
    try:
        resp = ollama_client.chat(
            model=VISION_MODEL,
            messages=[{"role": "user", "content": SINGLE_CHART_PROMPT, "images": [img_bytes]}],
            options={"num_predict": 1024, "temperature": 0},
        )
        raw = resp["message"]["content"].strip()
        clean = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
        try:
            parsed = json.loads(clean)
            lines = []
            title    = parsed.get("title") or ""
            subtitle = parsed.get("subtitle") or ""
            if title:    lines.append(f"TITLE: {title}")
            if subtitle: lines.append(f"PERIOD: {subtitle}")
            lines.append(f"TYPE: {parsed.get('type','').upper()}")
            data = parsed.get("data", {})
            for series in data.get("series", []):
                sname = series.get("name", "")
                for cat, val in series.get("values", {}).items():
                    lines.append(f"  {title} | {sname} | {cat}: {val}")
            headers = data.get("table_headers", [])
            for row in data.get("table_rows", []):
                row_parts = [f"{h}={row.get(h,'')}" for h in headers if h in row]
                if row_parts:
                    lines.append(f"  {title} | " + " | ".join(row_parts))
            for fact in parsed.get("key_facts", []):
                lines.append(f"FACT: {fact}")
            src = parsed.get("source_note") or ""
            if src: lines.append(f"SOURCE: {src}")
            result = "\n".join(lines)
            return result if result.strip() else raw
        except (json.JSONDecodeError, KeyError):
            return raw
    except Exception as e:
        logger.warning(f"  Single chart extraction failed: {e}")
        return "[Image description unavailable]"


def extract_equation(image: PIL.Image.Image) -> str:
    image = PIL.ImageOps.exif_transpose(image) or image
    image = image.convert("RGB")
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    img_bytes = buf.getvalue()
    try:
        resp = ollama_client.chat(
            model=VISION_MODEL,
            messages=[{"role": "user", "content": EQUATION_PROMPT, "images": [img_bytes]}],
            options={"num_predict": 512, "temperature": 0},
        )
        return resp["message"]["content"].strip()
    except Exception as e:
        logger.warning(f"  Equation extraction failed: {e}")
        return "[Mathematical equation — extraction unavailable]"


def describe_image_smart(image: PIL.Image.Image, ref: str = "") -> str:
    """Route each image to the correct extractor based on type classification."""
    img_type = classify_image(image)
    logger.info(f"    Image type detected: {img_type}  (ref={ref})")
    if img_type == "dual_chart":
        return "DUAL CHART EXTRACTION:\n" + extract_dual_chart(image)
    elif img_type == "colour_table":
        return "COLOUR-CODED TABLE EXTRACTION:\n" + extract_colour_table(image)
    elif img_type == "equation":
        return "MATHEMATICAL EQUATION:\n" + extract_equation(image)
    else:
        return "CHART/IMAGE EXTRACTION:\n" + extract_single_chart(image)

## CELL 7 │ Fix 3 — Box Section Preservation

In [7]:
from langchain_core.documents import Document

def extract_and_merge_boxes(
    text_docs: list[Document],
) -> tuple[list[Document], list[Document]]:
    """
    Scan text chunks for Box I.1, Box I.2, etc.
    Merge each box's scattered chunks into a single consolidated Document.
    Returns (non_box_docs, box_docs).
    """
    BOX_PATTERN = re.compile(r"Box\s+[IVX]+\.\d+[:]", re.IGNORECASE)

    box_groups: dict[str, list[Document]] = {}
    non_box_docs: list[Document] = []
    current_box_key: Optional[str] = None

    for doc in text_docs:
        text = doc.page_content
        box_match = BOX_PATTERN.search(text)
        if box_match:
            key = text[box_match.start():box_match.end()].strip().rstrip(":")
            current_box_key = key
            box_groups.setdefault(key, []).append(doc)
        elif current_box_key:
            is_new_section = (
                re.match(r"^\d+\.\d+\.", text.strip()) or
                re.match(r"^[A-Z][A-Z\s]{10,}$", text.strip())
            )
            if is_new_section:
                current_box_key = None
                non_box_docs.append(doc)
            else:
                box_groups[current_box_key].append(doc)
        else:
            non_box_docs.append(doc)

    box_docs: list[Document] = []
    for box_key, chunks in box_groups.items():
        merged_text = (
            f"BOX SECTION: {box_key}\n"
            f"{'='*60}\n"
            + "\n\n".join(c.page_content for c in chunks)
        )
        meta = chunks[0].metadata.copy()
        meta["type"]    = "box_section"
        meta["box_key"] = box_key
        box_docs.append(Document(page_content=merged_text, metadata=meta))
        logger.info(
            f"  Merged box '{box_key}' "
            f"({len(chunks)} chunks → 1 document, {len(merged_text)} chars)"
        )

    logger.info(
        f"  Box extraction: {len(box_docs)} boxes found, "
        f"{len(non_box_docs)} regular chunks remain"
    )
    return non_box_docs, box_docs

In [ ]:
%pip install "docling>=2.0.0"

## CELL 8 │ Document Conversion (Docling)

In [8]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

def convert_documents(sources: list[str]) -> dict:
    logger.info(f"Converting {len(sources)} PDF(s) with Docling...")
    converter = DocumentConverter(format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=PdfPipelineOptions(
                do_ocr=False,
                generate_picture_images=True,
            )
        ),
    })
    conversions = {}
    for source in sources:
        try:
            logger.info(f"  → {source}")
            conversions[source] = converter.convert(source=source).document
        except Exception as e:
            logger.error(f"  ✗ {source}: {e}")
    logger.info(f"  Done — {len(conversions)} document(s) converted")
    return conversions

## CELL 9 │ Chunk Extraction (Text / Tables / Images)

In [9]:
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.types.doc.document import TableItem


def extract_text_chunks(conversions: dict) -> list[Document]:
    """Extract and chunk text using Docling HybridChunker."""
    from transformers import AutoTokenizer
    try:
        tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
    except Exception:
        tokenizer = None

    texts, doc_id = [], 0
    for source, docling_doc in conversions.items():
        chunker = HybridChunker(tokenizer=tokenizer) if tokenizer else HybridChunker()
        for chunk in chunker.chunk(docling_doc):
            items = chunk.meta.doc_items
            if len(items) == 1 and isinstance(items[0], TableItem):
                continue
            refs = " ".join(item.get_ref().cref for item in items)
            texts.append(Document(
                page_content=chunk.text,
                metadata={
                    "doc_id": (doc_id := doc_id + 1),
                    "source": source,
                    "ref":    refs,
                    "type":   "text",
                },
            ))
    logger.info(f"  Text chunks   : {len(texts)}")
    return texts


def extract_table_chunks(
    conversions: dict, start_id: int = 0
) -> tuple[list[Document], list[Document]]:
    """
    Export tables as full Markdown parents + individual row children.
    Returns (parent_docs, child_docs).
    """
    from docling_core.types.doc.labels import DocItemLabel

    parent_docs, child_docs = [], []
    doc_id = start_id

    for source, docling_doc in conversions.items():
        for table in docling_doc.tables:
            if table.label not in [DocItemLabel.TABLE]:
                continue

            ref           = table.get_ref().cref
            full_markdown = table.export_to_markdown(doc=docling_doc)
            parent_id     = f"table_parent_{doc_id}"

            parent_docs.append(Document(
                page_content=full_markdown,
                metadata={
                    "doc_id":    (doc_id := doc_id + 1),
                    "parent_id": parent_id,
                    "source":    source,
                    "ref":       ref,
                    "type":      "table_parent",
                },
            ))

            lines       = full_markdown.strip().split("\n")
            header_line = lines[0] if lines else ""
            for row_line in lines[2:]:
                row_line = row_line.strip()
                if not row_line or row_line.startswith("|---"):
                    continue
                child_docs.append(Document(
                    page_content=(
                        f"Table: {ref}\n"
                        f"Headers: {header_line}\n"
                        f"Row: {row_line}"
                    ),
                    metadata={
                        "doc_id":    (doc_id := doc_id + 1),
                        "parent_id": parent_id,
                        "source":    source,
                        "ref":       ref,
                        "type":      "table_child",
                    },
                ))

    logger.info(
        f"  Table parents : {len(parent_docs)}"
        f"  │  Table rows (children) : {len(child_docs)}"
    )
    return parent_docs, child_docs


def extract_image_descriptions(
    conversions: dict, start_id: int = 0
) -> list[Document]:
    """Process each image through the smart router."""
    pictures, doc_id = [], start_id
    for source, docling_doc in conversions.items():
        pic_list = list(docling_doc.pictures)
        logger.info(f"  Processing {len(pic_list)} image(s) from {source[-40:]}")
        for i, picture in enumerate(pic_list, 1):
            image = picture.get_image(docling_doc)
            if not image:
                continue
            ref = picture.get_ref().cref
            logger.info(f"    [{i}/{len(pic_list)}] {ref}")
            description = describe_image_smart(image, ref=ref)
            pictures.append(Document(
                page_content=description,
                metadata={
                    "doc_id": (doc_id := doc_id + 1),
                    "source": source,
                    "ref":    ref,
                    "type":   "image",
                },
            ))
    logger.info(f"  Image descs   : {len(pictures)}")
    return pictures

## CELL 10 │ NEW — Multi-page Table Stitching

- Evaluator requirement: "Parser must maintain context across tables that
- split over multiple pages."
- Detects adjacent tables with similar titles (Table I.2a + I.2b) and merges
- them into a single stitched Document. Updates parent_table_map in-place.

In [10]:
def _normalise_title(text: str) -> str:
    """Strip table-part suffixes (a, b) and normalise whitespace."""
    text = re.sub(r"(Table\s+[IVX]+\.\d+)[a-z]", r"\1", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip().lower()

def _extract_title(markdown: str) -> str:
    for line in markdown.splitlines():
        line = line.strip()
        if line and not line.startswith("|"):
            return line
    return ""

def _titles_similar(t1: str, t2: str, threshold: float = 0.75) -> bool:
    n1, n2 = _normalise_title(t1), _normalise_title(t2)
    if n1 == n2:
        return True
    return SequenceMatcher(None, n1, n2).ratio() >= threshold


def stitch_multipage_tables(
    table_parents: list[Document],
    table_children: list[Document],
) -> tuple[list[Document], list[Document]]:
    """
    Detect and stitch multi-page tables (e.g. Table I.2a + Table I.2b).
    Returns updated (table_parents, table_children) lists.
    """
    if not table_parents:
        return table_parents, table_children

    children_by_parent: dict[str, list[Document]] = {}
    for child in table_children:
        pid = child.metadata.get("parent_id", "")
        children_by_parent.setdefault(pid, []).append(child)

    merged_parents: list[Document] = []
    merged_children_map: dict[str, list[Document]] = {}
    skip_next = False
    stitched_count = 0

    for i, doc in enumerate(table_parents):
        if skip_next:
            skip_next = False
            continue

        if i + 1 < len(table_parents):
            next_doc = table_parents[i + 1]
            t1 = _extract_title(doc.page_content)
            t2 = _extract_title(next_doc.page_content)

            if _titles_similar(t1, t2):
                stitched_content = (
                    f"[MULTI-PAGE TABLE — stitched from 2 pages]\n\n"
                    f"{doc.page_content}\n\n"
                    f"[Continued on next page]\n\n"
                    f"{next_doc.page_content}"
                )
                stitched_id = doc.metadata["parent_id"]
                stitched_doc = Document(
                    page_content=stitched_content,
                    metadata={
                        **doc.metadata,
                        "parent_id": stitched_id,
                        "type":      "table_parent",
                        "stitched":  True,
                    },
                )
                merged_parents.append(stitched_doc)

                kids_a = children_by_parent.get(doc.metadata["parent_id"], [])
                kids_b = children_by_parent.get(next_doc.metadata["parent_id"], [])
                kids_b_fixed = [
                    Document(page_content=k.page_content,
                             metadata={**k.metadata, "parent_id": stitched_id})
                    for k in kids_b
                ]
                merged_children_map[stitched_id] = kids_a + kids_b_fixed

                skip_next = True
                stitched_count += 1
                logger.info(f"  ✓ Stitched: '{t1[:50]}' + '{t2[:50]}'")
                continue

        merged_parents.append(doc)
        pid = doc.metadata["parent_id"]
        merged_children_map[pid] = children_by_parent.get(pid, [])

    merged_children: list[Document] = []
    for kids in merged_children_map.values():
        merged_children.extend(kids)

    logger.info(
        f"  Multi-page stitching: {stitched_count} pair(s) stitched. "
        f"Parents: {len(table_parents)} → {len(merged_parents)}"
    )
    return merged_parents, merged_children

In [ ]:
%pip install "langchain-chroma>=0.1.2"

## CELL 11 │ Vector Store — Hybrid BM25 + Dense + RRF

In [11]:
from langchain_chroma import Chroma

def build_hybrid_retriever(
    all_indexable_docs: list[Document],
    embeddings_model,
) -> tuple:
    """BM25 + Dense + RRF EnsembleRetriever."""
    from langchain_community.retrievers import BM25Retriever
    from langchain_classic.retrievers import EnsembleRetriever

    bm25_retriever = BM25Retriever.from_documents(all_indexable_docs, k=BM25_TOP_K)
    logger.info(f"  ✓ BM25 — {len(all_indexable_docs)} docs")

    dense_vectorstore = Chroma.from_documents(
        documents=all_indexable_docs,
        embedding=embeddings_model,
        collection_name="hybrid_dense_final",
        persist_directory=CHROMA_PERSIST_DIR + "_dense",
    )
    dense_retriever = dense_vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": DENSE_TOP_K},
    )
    logger.info("  ✓ Dense (ChromaDB) ready")

    ensemble = EnsembleRetriever(
        retrievers=[bm25_retriever, dense_retriever],
        weights=[0.4, 0.6],
    )
    logger.info("  ✓ Hybrid retriever (BM25 + Dense + RRF) ready")
    return ensemble, dense_vectorstore

In [ ]:
%pip instll "langchain-community>=0.3.0"

## CELL 12 │ FlashRank Cross-Encoder Reranker

In [12]:
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank
from langchain_classic.retrievers import ContextualCompressionRetriever

def build_compression_retriever(base_retriever) -> ContextualCompressionRetriever:
    reranker = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2", top_n=RERANKER_TOP_N)
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=reranker,
        base_retriever=base_retriever,
    )
    logger.info(f"  ✓ Reranker (ms-marco-MiniLM-L-12-v2) — keeps top {RERANKER_TOP_N}")
    return compression_retriever

## CELL 13 │ Parent Lookup (table children → full table)

In [14]:
def resolve_to_parents(
    docs: list[Document],
    parent_table_map: dict[str, Document],
) -> list[Document]:
    """Replace table_child hits with their full parent table Documents."""
    resolved = []
    seen_ids = set()
    for doc in docs:
        dtype     = doc.metadata.get("type", "")
        parent_id = doc.metadata.get("parent_id")
        if dtype == "table_child" and parent_id and parent_id in parent_table_map:
            parent_doc = parent_table_map[parent_id]
            pid = parent_doc.metadata.get("parent_id", parent_id)
            if pid not in seen_ids:
                seen_ids.add(pid)
                resolved.append(parent_doc)
        else:
            uid = doc.metadata.get("doc_id", id(doc))
            if uid not in seen_ids:
                seen_ids.add(uid)
                resolved.append(doc)
    return resolved

## CELL 14 │ RAG Chain with Grounded Prompt

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

SYSTEM_PROMPT = """\
You are a precise financial and economic analyst assistant specialising in \
Indian economic data from government reports.

INSTRUCTIONS:
1. Answer using ONLY the information in the CONTEXT below.
2. Cite the context block using [1], [2], [3] format for every figure you state.
3. Quote exact numbers as they appear — never round or approximate.
4. For colour-coded table data, include the performance signal when relevant
   (e.g. "above historical average" or "below historical average").
5. For box sections (nowcasting, methodology), cite them specifically.
6. If the answer is not in the context, say exactly:
   "The provided documents do not contain sufficient information to answer this question."
7. Do NOT use any external knowledge or make assumptions.
8. Keep answers concise and factual.

CONTEXT:
{context}
"""


def format_docs_with_numbers(docs: list[Document]) -> str:
    parts = []
    for i, doc in enumerate(docs, 1):
        dtype  = doc.metadata.get("type", "text").upper()
        source = doc.metadata.get("source", "")[-50:]
        score  = doc.metadata.get("relevance_score", None)
        score_str = f" score={score:.3f}" if score is not None else ""
        if dtype == "BOX_SECTION":
            label = f"[{i}] [📦 BOX SECTION{score_str}]"
        elif dtype == "IMAGE" and "COLOUR-CODED TABLE" in doc.page_content.upper():
            label = f"[{i}] [🎨 COLOUR TABLE{score_str}]"
        elif dtype == "IMAGE" and "DUAL CHART" in doc.page_content.upper():
            label = f"[{i}] [📊 DUAL CHART{score_str}]"
        else:
            label = f"[{i}] [{dtype}{score_str}]"
        parts.append(f"{label} (source: ...{source})\n{doc.page_content}")
    return "\n\n" + "\n\n".join(parts)


def build_rag_chain(compression_retriever, parent_table_map: dict, chat_model):
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "{question}"),
    ])

    def retrieve_and_resolve(query: str) -> str:
        reranked = compression_retriever.invoke(query)
        final    = resolve_to_parents(reranked, parent_table_map)
        return format_docs_with_numbers(final)

    rag_chain = (
        {
            "context":  RunnableLambda(retrieve_and_resolve),
            "question": RunnablePassthrough(),
        }
        | prompt
        | chat_model
        | StrOutputParser()
    )
    logger.info("  ✓ RAG chain ready")
    return rag_chain

## CELL 15 │ Full Pipeline Builder

In [17]:
def build_pipeline(skip_images: bool = False):
    """
    Orchestrate the complete indexing pipeline.

    Steps:
      1.  Convert PDFs with Docling
      2.  Extract text chunks
      3.  Detect and merge box sections (Fix 3)
      4.  Extract table chunks (parent + child)
      5.  NEW: Stitch multi-page tables (Table I.2a + I.2b)
      6.  Extract images via smart router (Fix 1 + Fix 2)
      7.  Build hybrid retriever (BM25 + Dense + RRF)
      8.  Attach FlashRank reranker
      9.  Build RAG chain

    Returns: (rag_chain, compression_retriever, parent_table_map)
    """
    print("\n" + "═"*72)
    print("  BUILDING PIPELINE  —  Final Version with All Fixes")
    print("═"*72)

    # ── Step 1: Convert PDFs ──────────────────────────────────────────────
    conversions = convert_documents(PDF_SOURCES)
    assert conversions, "No documents converted. Check PDF_SOURCES."

    # ── Step 2: Extract text chunks ───────────────────────────────────────
    print("\n📄 Extracting document chunks...")
    text_docs = extract_text_chunks(conversions)

    # ── Step 3: Detect and merge box sections ─────────────────────────────
    print("\n📦 Extracting Box sections (Fix 3)...")
    text_docs, box_docs = extract_and_merge_boxes(text_docs)
    print(f"   Box sections found: {len(box_docs)}")
    for bd in box_docs:
        print(f"   └─ {bd.metadata.get('box_key','?')} ({len(bd.page_content)} chars)")

    # ── Step 4: Extract tables ────────────────────────────────────────────
    table_parents, table_children = extract_table_chunks(
        conversions,
        start_id=len(text_docs) + len(box_docs),
    )

    # ── Step 5: Stitch multi-page tables ─────────────────────────────────
    print("\n📎 Stitching multi-page tables...")
    table_parents, table_children = stitch_multipage_tables(
        table_parents, table_children
    )
    for doc in table_parents:
        flag = " ← STITCHED" if doc.metadata.get("stitched") else ""
        print(f"   • {_extract_title(doc.page_content)[:60]}{flag}")

    # ── Step 6: Extract images ────────────────────────────────────────────
    if skip_images:
        image_docs = []
        logger.info("  Image processing skipped (skip_images=True)")
    else:
        print("\n🖼️  Extracting images with smart router (Fix 1 + Fix 2)...")
        image_docs = extract_image_descriptions(
            conversions,
            start_id=(len(text_docs) + len(box_docs)
                      + len(table_parents) + len(table_children)),
        )

    print(f"\n  Chunk Summary:")
    print(f"    Text docs        : {len(text_docs)}")
    print(f"    Box sections     : {len(box_docs)}   ← Fix 3")
    print(f"    Table parents    : {len(table_parents)}  (after stitching)")
    print(f"    Table children   : {len(table_children)}")
    print(f"    Image docs       : {len(image_docs)}  ← Fix 1 + Fix 2")

    # ── Build parent table map ────────────────────────────────────────────
    parent_table_map: dict[str, Document] = {
        doc.metadata["parent_id"]: doc
        for doc in table_parents
    }

    # ── Step 7: Build hybrid retriever ───────────────────────────────────
    print("\n🔍 Building hybrid retriever...")
    indexable_docs = text_docs + box_docs + table_children + image_docs
    ensemble_retriever, _ = build_hybrid_retriever(indexable_docs, embeddings_model)

    # ── Step 8: Attach reranker ───────────────────────────────────────────
    print("\n⚡ Attaching cross-encoder reranker...")
    compression_retriever = build_compression_retriever(ensemble_retriever)

    # ── Step 9: Build RAG chain ───────────────────────────────────────────
    print("\n🔗 Building RAG chain...")
    rag_chain = build_rag_chain(compression_retriever, parent_table_map, chat_model)

    print("\n" + "═"*72)
    print("  PIPELINE READY")
    print("═"*72)

    return rag_chain, compression_retriever, parent_table_map


# Build
rag_chain, compression_retriever, parent_table_map = build_pipeline(skip_images=False)

02:47:11 [INFO] Converting 1 PDF(s) with Docling...
02:47:11 [INFO]   → C:\Users\Rutvik Rathva\Downloads\Intern 3.12\Dataset_removed-pages-deleted.pdf
02:47:11 [INFO] detected formats: [<InputFormat.PDF: 'pdf'>]
02:47:11 [INFO] Going to convert document batch...
02:47:11 [INFO] Initializing pipeline for StandardPdfPipeline with options hash 6711d9b92950a20d5f7ad32bdd26c5ab
02:47:11 [INFO] Accelerator device: 'cpu'



════════════════════════════════════════════════════════════════════════
  BUILDING PIPELINE  —  Final Version with All Fixes
════════════════════════════════════════════════════════════════════════


02:47:12 [INFO] Accelerator device: 'cpu'
02:47:13 [INFO] Processing document Dataset_removed-pages-deleted.pdf
02:47:26 [ERROR] Stage preprocess failed for run 1, pages [26]: std::bad_alloc
02:47:27 [ERROR] Stage preprocess failed for run 1, pages [27]: std::bad_alloc
02:47:28 [ERROR] Stage preprocess failed for run 1, pages [28]: std::bad_alloc
02:49:05 [INFO] Finished converting document Dataset_removed-pages-deleted.pdf in 114.01 sec.
02:49:05 [INFO]   Done — 1 document(s) converted



📄 Extracting document chunks...


Token indices sequence length is longer than the specified maximum sequence length for this model (1511 > 512). Running this sequence through the model will result in indexing errors
02:49:07 [INFO]   Text chunks   : 61
02:49:07 [INFO]   Box extraction: 0 boxes found, 61 regular chunks remain
02:49:08 [INFO]   Table parents : 6  │  Table rows (children) : 68
02:49:08 [INFO]   Multi-page stitching: 0 pair(s) stitched. Parents: 6 → 6
02:49:08 [INFO]   Processing 16 image(s) from n 3.12\Dataset_removed-pages-deleted.pdf
02:49:08 [INFO]     [1/16] #/pictures/0
02:49:08 [INFO]     Image type detected: single_chart  (ref=#/pictures/0)


...


  Chunk Summary:
    Text docs        : 61
    Box sections     : 0   ← Fix 3
    Table parents    : 6  (after stitching)
    Table children   : 68
    Image docs       : 16  ← Fix 1 + Fix 2

🔍 Building hybrid retriever...


02:55:21 [INFO] HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
02:55:22 [INFO]   ✓ Dense (ChromaDB) ready
02:55:22 [INFO]   ✓ Hybrid retriever (BM25 + Dense + RRF) ready



⚡ Attaching cross-encoder reranker...


02:55:24 [INFO]   ✓ Reranker (ms-marco-MiniLM-L-12-v2) — keeps top 5
02:55:24 [INFO]   ✓ RAG chain ready



🔗 Building RAG chain...

════════════════════════════════════════════════════════════════════════
  PIPELINE READY
════════════════════════════════════════════════════════════════════════


## CELL 16 │ NEW — Semantic Query Understanding Layer

- Evaluator requirement: "A separate semantic understanding layer on top of
- the retrieval process to ensure meaningful matches rather than just
- simple string matching."
- Implementation:
- HyDE  — generate a short hypothetical answer, embed in answer-space
- Query Expansion — produce 2 rephrasings for diverse vocabulary coverage

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

_semantic_llm =ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0.2,
    num_ctx=2048,
)

_hyde_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an economic analyst. Write a SHORT factual answer (2–3 sentences) "
     "to the question below, as if it appeared in the India Economic Survey. "
     "Include likely numbers, percentages, and technical terms. "
     "Do NOT say you don't know — always produce a plausible answer."),
    ("human", "{question}"),
])
_hyde_chain = _hyde_prompt | _semantic_llm | StrOutputParser()

_expansion_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Generate exactly 2 alternative phrasings of the question below. "
     "Each should use different vocabulary and perspective but ask the same thing. "
     "Output ONLY the 2 questions, one per line, no numbering, no explanation."),
    ("human", "{question}"),
])
_expansion_chain = _expansion_prompt | _semantic_llm | StrOutputParser()


def semantic_query_layer(query: str) -> list[str]:
    """
    Convert a single query into up to 4 semantically diverse retrieval queries:
      [0] original query
      [1] expansion variant 1
      [2] expansion variant 2
      [3] HyDE hypothetical answer
    """
    logger.info(f"  Semantic layer: generating variants for: {query[:60]}...")
    queries = [query]

    try:
        expansion_raw = _expansion_chain.invoke({"question": query})
        expansions = [
            line.strip()
            for line in expansion_raw.strip().splitlines()
            if line.strip() and len(line.strip()) > 10
        ][:2]
        queries.extend(expansions)
        logger.info(f"    Query expansions: {len(expansions)}")
    except Exception as e:
        logger.warning(f"    Query expansion failed: {e}")

    try:
        hyde_answer = _hyde_chain.invoke({"question": query})
        if hyde_answer and len(hyde_answer.strip()) > 20:
            queries.append(hyde_answer.strip())
            logger.info(f"    HyDE answer generated: {hyde_answer[:80]}...")
    except Exception as e:
        logger.warning(f"    HyDE generation failed: {e}")

    logger.info(f"  Semantic layer: {len(queries)} query variants ready")
    return queries


def semantic_retrieve_and_rerank(
    query: str,
    compression_retriever,
    parent_table_map: dict,
    top_n: int = 5,
) -> list[Document]:
    """
    Full semantic retrieval:
      1. Generate query variants via semantic_query_layer()
      2. Run compression_retriever on each variant
      3. Deduplicate by content hash
      4. Apply parent-table resolution
      5. Sort by reranker score, return top_n
    """
    import hashlib

    all_docs: list[Document] = []
    seen_hashes: set[str] = set()

    for variant in semantic_query_layer(query):
        try:
            docs = compression_retriever.invoke(variant)
            for doc in docs:
                h = hashlib.md5(doc.page_content[:200].encode()).hexdigest()
                if h not in seen_hashes:
                    seen_hashes.add(h)
                    all_docs.append(doc)
        except Exception as e:
            logger.warning(f"    Retrieval failed for variant: {e}")

    resolved = resolve_to_parents(all_docs, parent_table_map)
    resolved.sort(key=lambda d: d.metadata.get("relevance_score", 0.0), reverse=True)
    final = resolved[:top_n]
    logger.info(
        f"  Semantic retrieve: {len(all_docs)} raw → "
        f"{len(resolved)} deduped → {len(final)} final"
    )
    return final


def build_semantic_rag_chain(compression_retriever, parent_table_map: dict, chat_model):
    """RAG chain that uses semantic_retrieve_and_rerank() for every query."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "{question}"),
    ])

    def retrieve_and_format(query: str) -> str:
        final_docs = semantic_retrieve_and_rerank(
            query, compression_retriever, parent_table_map
        )
        return format_docs_with_numbers(final_docs)

    rag_chain = (
        {
            "context":  RunnableLambda(retrieve_and_format),
            "question": RunnablePassthrough(),
        }
        | prompt
        | chat_model
        | StrOutputParser()
    )
    logger.info("  ✓ Semantic RAG chain ready (HyDE + Query Expansion + Reranker)")
    return rag_chain


# Wire up semantic chain
print("\n🧠 Building Semantic Query Layer (HyDE + Query Expansion)...")
semantic_rag_chain = build_semantic_rag_chain(
    compression_retriever, parent_table_map, chat_model
)
print("  ✓ Semantic RAG chain ready")
print("\n  Components active:")
print("    [1] HyDE  — generates a hypothetical answer, embeds in answer-space")
print("    [2] Query Expansion — 2 rephrased variants per query")
print("    [3] FlashRank cross-encoder reranker — rescores all candidates")
print("    [4] Parent-table resolution — child row → full table")

# Quick test
test_q = "By how much did the weighted average lending rate (WALR) on fresh Rupee loans decline between February and November 2025?"
print(f"\n🔬 Semantic layer test: '{test_q}'")
variants = semantic_query_layer(test_q)
print(f"\n  Query variants generated ({len(variants)}):")
for i, v in enumerate(variants, 1):
    label = ["Original", "Expansion 1", "Expansion 2", "HyDE Answer"][min(i-1, 3)]
    print(f"    [{i}] {label}: {v[:100]}")

02:56:54 [INFO]   ✓ Semantic RAG chain ready (HyDE + Query Expansion + Reranker)
02:56:54 [INFO]   Semantic layer: generating variants for: By how much did the weighted average lending rate (WALR) on ...



🧠 Building Semantic Query Layer (HyDE + Query Expansion)...
  ✓ Semantic RAG chain ready

  Components active:
    [1] HyDE  — generates a hypothetical answer, embeds in answer-space
    [2] Query Expansion — 2 rephrased variants per query
    [3] FlashRank cross-encoder reranker — rescores all candidates
    [4] Parent-table resolution — child row → full table

🔬 Semantic layer test: 'By how much did the weighted average lending rate (WALR) on fresh Rupee loans decline between February and November 2025?'


02:57:01 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
02:57:02 [INFO]     Query expansions: 2
02:57:02 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
02:57:04 [INFO]     HyDE answer generated: According to the Reserve Bank of India's (RBI) data, the weighted average lendin...
02:57:04 [INFO]   Semantic layer: 4 query variants ready



  Query variants generated (4):
    [1] Original: By how much did the weighted average lending rate (WALR) on fresh Rupee loans decline between Februa
    [2] Expansion 1: What percentage decrease in the weighted average lending rate on fresh Rupee loans occurred between 
    [3] Expansion 2: How much did the weighted average lending rate on fresh Rupee loans fall from February to November 2
    [4] HyDE Answer: According to the Reserve Bank of India's (RBI) data, the weighted average lending rate (WALR) on fre


## CELL 17 │ Query Helper (uses semantic_rag_chain)

In [30]:
def ask(query: str, show_context: bool = True) -> str:
    """Run a single RAG query with full context display."""
    print(f"\n{'─'*72}")
    print(f"📌 QUERY: {query}")

    if show_context:
        reranked = compression_retriever.invoke(query)
        resolved = resolve_to_parents(reranked, parent_table_map)
        print(f"\n📚 Context ({len(resolved)} chunks → reranked + parent-resolved):")
        for i, doc in enumerate(resolved, 1):
            dtype = doc.metadata.get("type", "?").upper()
            score = doc.metadata.get("relevance_score", "n/a")
            flag = ""
            if dtype == "BOX_SECTION":                       flag = " 📦"
            elif dtype == "IMAGE":
                content = doc.page_content.upper()
                if "DUAL CHART"     in content:              flag = " 📊"
                elif "COLOUR-CODED" in content:              flag = " 🎨"
                else:                                        flag = " 🖼️"
            elif dtype == "TABLE_PARENT":                    flag = " 📋"
            print(f"\n  [{i}] {dtype}{flag} | score={score}")
            print(f"       {doc.page_content[:300]}...")

    answer = semantic_rag_chain.invoke(query)
    print(f"\n🟦 ANSWER:\n{answer}")
    print("─"*72)
    return answer


# Single test
DEFAULT_QUERY="Q15: What structural fundamentals are expected to guide the exchange rate dynamics of the Indian Rupee over the medium to long term, rather than short-term market fluctuations?"
ask(DEFAULT_QUERY)


────────────────────────────────────────────────────────────────────────
📌 QUERY: Q15: What structural fundamentals are expected to guide the exchange rate dynamics of the Indian Rupee over the medium to long term, rather than short-term market fluctuations?


03:46:57 [INFO] HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"



📚 Context (5 chunks → reranked + parent-resolved):

  [1] TEXT | score=0.10039818286895752
       The  chapter  reviews  key  macroeconomic  fundamentals,  including  inflation, financial sector conditions, fiscal developments, external sector performance, and labour market trends. Inflation has moderated, financial sector balance sheets remain healthy, and fiscal policy continues to strike a ba...

  [2] TEXT | score=0.011796788312494755
       space, and long-term economic sovereignty. This mechanics are detailed in Chapter 16 (Part-I) and (Part-II) of this survey.
- 1.7. Against this global backdrop, this chapter reviews the performance of the Indian economy as reflected in the First Advance Estimates for FY26. It analyses the demand and...

  [3] TEXT | score=0.001356214634142816
       The global  economic  environment  remains  uncertain,  shaped  by  geopolitical tensions,  trade  disruptions,  and  divergent  growth  and  inflation  outcomes across major economies. While globa

03:46:59 [INFO]   Semantic layer: generating variants for: Q15: What structural fundamentals are expected to guide the ...
03:47:05 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
03:47:06 [INFO]     Query expansions: 2
03:47:06 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
03:47:08 [INFO]     HyDE answer generated: The exchange rate dynamics of the Indian Rupee (INR) are expected to be guided b...
03:47:08 [INFO]   Semantic layer: 4 query variants ready
03:47:10 [INFO] HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
03:47:10 [INFO] HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
03:47:11 [INFO] HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
03:47:12 [INFO] HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
03:47:13 [INFO]   Semantic retrieve: 8 raw → 8 deduped → 5 final
03:47:19 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1


🟦 ANSWER:
The provided documents do not contain sufficient information to answer this question.
────────────────────────────────────────────────────────────────────────


'The provided documents do not contain sufficient information to answer this question.'

## CELL 18 │ Batch Query — 15 Factual + 10 Semantic Questions

In [ ]:
# factual_queries = [
#     "1.  What is the real GDP growth rate for FY26 according to the First Advance Estimates?",
#     "2.  What is the GVA growth rate estimated for FY26?",
#     "3.  What share of GDP does PFCE account for in FY26?",
#     "4.  What is the estimated share of Gross Fixed Capital Formation (GFCF) in GDP in FY26?",
#     "5.  By how much did FDI flows decline globally in 2024 (YoY), excluding conduit economies?",
#     "6.  What was the manufacturing growth rate in H1 FY26?",
#     "7.  What is the estimated growth rate of the services sector in FY26?",
#     "8.  What growth rate is estimated for agriculture and allied services in FY26?",
#     "9.  What was the growth of exports of goods and services in H1 FY26?",
#     "10. What is the nowcasted real GDP growth rate for Q3 FY26?",
#     "11. What model is used for nowcasting GDP growth?",
#     "12. How many high-frequency indicators were used in the nowcasting model?",
#     "13. What was the growth in non-food bank credit in Q3 FY26?",
#     "14. What was capacity utilisation in Q2 FY26?",
#     "15. What percentage of rural households reported increased consumption in the NABARD survey?",
# ]

# semantic_queries = [
#     "1.  Why is global economic growth described as 'fragile and diverging'?",
#     "2.  Why did global economic activity remain stable in the short term despite US tariff impositions?",
#     "3.  How has domestic demand supported India's GDP growth in FY26?",
#     "4.  Why has manufacturing's share in nominal GVA declined even though real output remains stable?",
#     "5.  How do high-frequency indicators help in assessing near-term GDP growth?",
#     "6.  Why is the services sector described as the stabilising component of GVA?",
#     "7.  What structural factors explain the volatility in agricultural growth?",
#     "8.  Why is the investment cycle said to be strengthening in FY26?",
#     "9.  How do geopolitical tensions contribute to the resurgence of economic statecraft?",
#     "10. What is meant by 'strategic indispensability,' and why is it important for India?",
# ]

# print("\n\n" + "═"*72)
# print("  BATCH QUERY — 15 Factual Questions")
# print("═"*72)
# results_factual = {}
# for q in factual_queries:
#     results_factual[q] = ask(q, show_context=True)

# print("\n\n" + "═"*72)
# print("  BATCH QUERY — 10 Semantic Questions")
# print("═"*72)
# results_semantic = {}
# for q in semantic_queries:
#     results_semantic[q] = ask(q, show_context=True)

# # ── Summary ───────────────────────────────────────────────────────────────────
# print("\n\n" + "═"*72)
# print("  ANSWER SUMMARY")
# print("═"*72)
# for q, a in {**results_factual, **results_semantic}.items():
#     print(f"\n{q}")
#     print(f"   → {a[:200].replace(chr(10), ' ')}")
# print("═"*72)

## CELL 20 | Notebook Chat UI (No API)

- Run this after the pipeline setup cells.
- If needed once: `%pip install -U gradio`
- Launches a local chat interface so users can ask questions directly.

In [33]:
%pip install -U gradio


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gradio as gr
def _ensure_chat_ready() -> None:
    required = [
        "compression_retriever",
        "parent_table_map",
        "semantic_rag_chain",
        "semantic_retrieve_and_rerank",
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            "Missing required objects: " + ", ".join(missing) +
            ". Run pipeline + semantic cells first."
        )




def _clean_answer_text(text: str) -> str:
    # Remove leading hedging/context prefaces.
    text = re.sub(r"(?is)^\s*according to (the )?(provided )?context\s*,?\s*", "", text)
    text = re.sub(r"(?is)^\s*based on (the )?(provided )?context\s*,?\s*", "", text)

    # Remove patterns like "According to [3]," / "According to {3},".
    text = re.sub(r"(?i)\baccording to\s*[\[{(]\s*\d+\s*[\]})]\s*,?\s*", "", text)

    # Remove explicit source snippets like (Source: 1.24) anywhere.
    text = re.sub(r"\s*\(\s*source\s*:\s*[^)]*\)", "", text, flags=re.IGNORECASE)

    # Remove inline "Source: ..." tails.
    text = re.sub(r"(?i)\s*source\s*:\s*[^.\n]*(?:[.\n]|$)", ". ", text)

    # Remove bare numeric citation tokens [3], {3}, (3).
    text = re.sub(r"\s*[\[{(]\s*\d+\s*[\]})]", "", text)

    # Normalize whitespace/punctuation.
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\s*\.\s*\.", ".", text)
    return text


def _format_context(docs, max_chars: int = 350) -> str:
    lines = []
    for i, doc in enumerate(docs, start=1):
        dtype = str(doc.metadata.get("type", "text")).upper()
        score = doc.metadata.get("relevance_score")
        source = str(doc.metadata.get("source", ""))
        snippet = doc.page_content[:max_chars].replace("", " ").strip()
        score_txt = f"{score:.3f}" if isinstance(score, (int, float)) else "n/a"
        lines.append(f"{i}. [{dtype}] score={score_txt} source={source}{snippet}")
    return "".join(lines)


def chat_response(message, history, top_n, show_context):
    _ensure_chat_ready()

    query = (message or "").strip()
    if not query:
        return "Please enter a question."

    docs = semantic_retrieve_and_rerank(
        query=query,
        compression_retriever=compression_retriever,
        parent_table_map=parent_table_map,
        top_n=int(top_n),
    )
    answer = semantic_rag_chain.invoke(query)
    answer = _clean_answer_text(str(answer))

    if not show_context:
        return answer

    context_block = _format_context(docs)
    return f"{answer}---Retrieved context:{context_block}"


with gr.Blocks(title="IBM RAG Chat") as demo:
    gr.Markdown("# IBM RAG ChatAsk questions over your indexed document.")
    gr.ChatInterface(
        fn=chat_response,
        # additional_inputs=[
        #     gr.Slider(1, 10, value=4, step=1, label="Top-K context"),
        #     gr.Checkbox(value=True, label="Show retrieved context"),
        # ],
    )

# Set share=True only if you need a public temporary link.
demo.launch(share=False, inbrowser=True)


c:\Users\Rutvik Rathva\Downloads\Modular Code\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Rutvik Rathva\Downloads\Modular Code\myvenv\Lib\site-packages\gradio\utils.py:1177: UserWarning: Expected 4 arguments for function <function chat_response at 0x0000016D6C9E6980>, received 2.
  warnings.warn(
c:\Users\Rutvik Rathva\Downloads\Modular Code\myvenv\Lib\site-packages\gradio\utils.py:1181: UserWarning: Expected at least 4 arguments for function <function chat_response at 0x0000016D6C9E6980>, received 2.
  warnings.warn(
c:\Users\Rutvik Rathva\Downloads\Modular Code\myvenv\Lib\site-packages\gradio\utils.py:1177: UserWarning: Expected 4 arguments for function <function chat_response at 0x0000016D6CB2F2E0>, received 2.
  warnings.warn(
c:\Users\Rutvik Rathva\Downloads\Modular Code\myvenv\Lib\site-pa

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\Rutvik Rathva\Downloads\Modular Code\myvenv\Lib\site-packages\gradio\helpers.py:1083: UserWarning: Unexpected argument. Filling with None.
  warnings.warn("Unexpected argument. Filling with None.")
Traceback (most recent call last):
  File "c:\Users\Rutvik Rathva\Downloads\Modular Code\myvenv\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rutvik Rathva\Downloads\Modular Code\myvenv\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rutvik Rathva\Downloads\Modular Code\myvenv\Lib\site-packages\gradio\blocks.py", line 2158, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rutvik Rathva\Downloads\Modular Code\myvenv\Lib\site-packages\gradio\blocks.py